# BetaTrax Sprint 1 Demo

This notebook runs the Sprint 1 demo flow against `https://betatrax.yeelam.site` using `curl` and prints the returned JSON at each step.

Notes for this deployed build:
- Course demo severity `Minor` is mapped here to API value `Low`.
- Course demo priority `High` is mapped here to API value `P1`.
- This build includes a dedicated `GET /api/defects/<id>/` endpoint for the detail steps in the demo flow.

Run the notebook from top to bottom.

## Setup Block 1: Demo Configuration

Purpose: Define the shared configuration used by all later test cases.

This block defines:
- the demo site base URL: `https://betatrax.yeelam.site`
- which `curl` binary to use
- Product Owner and Developer login credentials
- the target defect payload that will be submitted in Test Case 1
- the email address used for this demo: `tester@gmail.com`
- the mapped severity and priority values used later in the accept step
- a variable `target_report_id` that will store the newly created defect ID after submission

Expected: This block only defines variables. It does not call the API yet.

In [ ]:
import json
import os
import subprocess
import tempfile
from urllib.parse import urlencode

BASE_URL = "https://betatrax.yeelam.site"
CURL_BIN = "curl.exe" if os.name == "nt" else "curl"

OWNER_AUTH = ("owner-001", "Pass1234!")
DEVELOPER_AUTH = ("dev-001", "Pass1234!")

TARGET_EMAIL = "tester@gmail.com"
TARGET_PAYLOAD = {
    "product_id": "Prod_1",
    "version": "0.9.0",
    "title": "Hit count incorrect",
    "description": "Following a successful search, the hit count is different to the number of matches displayed.",
    "steps": "1. Enter search criteria that ensure at least one match. 2. Search. 3. Compare matches displayed with the number of hits reported.",
    "tester_id": "Tester_1",
    "email": TARGET_EMAIL,
}

DEMO_SEVERITY = "Low"
DEMO_PRIORITY = "P1"

target_report_id = None


## Setup Block 2: Helper Functions

Purpose: Define reusable helper functions for the demo.

This block defines:
- `curl_json(...)`: sends a `curl` request, captures the HTTP status, parses the returned JSON, and prints one JSON envelope containing request and response details
- `show_target_from_list(...)`: takes a list response, filters out the target defect by `report_id`, and prints only that defect as JSON for list-based verification steps

Expected: This block only defines functions. It does not call the API yet.

In [ ]:
def curl_json(method, path, payload=None, auth=None, params=None, label=None):
    url = BASE_URL + path
    if params:
        url += "?" + urlencode(params)

    cmd = [
        CURL_BIN,
        "-sS",
        "-X",
        method.upper(),
        url,
        "-H",
        "Accept: application/json",
        "-w",
        "\n__HTTP_STATUS__:%{http_code}",
    ]

    if auth:
        cmd.extend(["-u", f"{auth[0]}:{auth[1]}"])

    temp_path = None
    if payload is not None:
        with tempfile.NamedTemporaryFile("w", delete=False, suffix=".json", encoding="utf-8") as temp_file:
            json.dump(payload, temp_file, ensure_ascii=False)
            temp_path = temp_file.name
        cmd.extend([
            "-H",
            "Content-Type: application/json",
            "--data-binary",
            f"@{temp_path}",
        ])

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
    finally:
        if temp_path and os.path.exists(temp_path):
            os.unlink(temp_path)

    raw_output = result.stdout.strip()
    body_text, _, status_text = raw_output.rpartition("\n__HTTP_STATUS__:")
    http_status = int(status_text) if status_text else None

    try:
        body = json.loads(body_text)
    except json.JSONDecodeError:
        body = body_text

    output = {
        "label": label,
        "request": {
            "method": method.upper(),
            "url": url,
            "payload": payload,
        },
        "status_code": http_status,
        "stderr": result.stderr.strip() or None,
        "body": body,
    }
    print(json.dumps(output, indent=2, ensure_ascii=False))

    return output


def show_target_from_list(response, report_id, label):
    items = response["body"].get("items", [])
    target = next((item for item in items if item.get("report_id") == report_id), None)
    output = {
        "label": label,
        "report_id": report_id,
    }
    if target is None:
        output["body"] = {"error": f"Report {report_id} not found in response items."}
    else:
        output["body"] = target
    print(json.dumps(output, indent=2, ensure_ascii=False))
    return target


## Test Case 1: Submit Defect Report

Purpose: Demonstrate `US-01` and `US-02` from `BT-UC2`.

Actor: Beta Tester

Input:
- Product ID: `Prod_1`
- Version: `0.9.0`
- Title: `Hit count incorrect`
- Description: following a successful search, the hit count is different to the number of matches displayed
- Steps: enter search criteria, search, compare displayed matches with hit count
- Tester ID: `Tester_1`
- Email: `tester@gmail.com`

Expected: API returns `201`, creates a new defect, and returns `report_id` with status `New`.

In [ ]:
step1 = curl_json(
    "POST",
    "/api/defects/new/",
    payload=TARGET_PAYLOAD,
    label="Test Case 1. Submit defect report",
)
target_report_id = step1["body"]["report_id"]
print(json.dumps({"label": "Target defect report selected", "report_id": target_report_id}, indent=2, ensure_ascii=False))


## Test Case 2: List New Defects

Purpose: Demonstrate `US-04` from `BT-UC3`.

Actor: Product Owner

Request:
- Endpoint: `GET /api/defects/?status=New`
- Auth: `owner-001`

Expected: Response contains only owner-visible defects in status `New`, including the target defect created in Test Case 1.

In [ ]:
step2 = curl_json(
    "GET",
    "/api/defects/",
    auth=OWNER_AUTH,
    params={"status": "New"},
    label="Test Case 2. List New defects for Product Owner",
)


## Test Case 3: View Target Defect Details While New

Purpose: Show the detailed JSON for the target defect before acceptance.

Actor: Product Owner

Request:
- Endpoint: `GET /api/defects/<report_id>/`
- Auth: `owner-001`

Expected: Printed JSON shows the target defect with status `New`.

In [ ]:
step3 = curl_json(
    "GET",
    f"/api/defects/{target_report_id}/",
    auth=OWNER_AUTH,
    label="Test Case 3. View target defect details while New",
)


## Test Case 4: Accept New Defect as Open

Purpose: Demonstrate `US-05` and `US-06` from `BT-UC3`.

Actor: Product Owner

Input:
- Action: `accept_open`
- Severity: `Low`  
  Course demo says `Minor`; this deployed API accepts `High | Medium | Low`, so `Low` is used.
- Priority: `P1`  
  Course demo says `High`; this deployed API accepts `P1 | P2 | P3`, so `P1` is used.

Expected: API returns `200` and the target defect status becomes `Open`.

In [ ]:
step4 = curl_json(
    "POST",
    f"/api/defects/{target_report_id}/actions/",
    auth=OWNER_AUTH,
    payload={
        "action": "accept_open",
        "severity": DEMO_SEVERITY,
        "priority": DEMO_PRIORITY,
    },
    label="Test Case 4. Accept target defect report",
)


## Test Case 5: List Open Defects

Purpose: Demonstrate `US-07` from `BT-UC4`.

Actor: Developer

Request:
- Endpoint: `GET /api/defects/?status=Open`
- Auth: `dev-001`

Expected: Response contains developer-visible defects in status `Open`, including the target defect after acceptance.

In [ ]:
step5 = curl_json(
    "GET",
    "/api/defects/",
    auth=DEVELOPER_AUTH,
    params={"status": "Open"},
    label="Test Case 5. List Open defects for Developer",
)


## Test Case 6: View Target Defect Details While Open

Purpose: Show the target defect JSON after it has been accepted and moved to `Open`.

Actor: Developer

Request:
- Endpoint: `GET /api/defects/<report_id>/`
- Auth: `dev-001`

Expected: Printed JSON shows the target defect with status `Open`, severity `Low`, and priority `P1`.

In [ ]:
step6 = curl_json(
    "GET",
    f"/api/defects/{target_report_id}/",
    auth=DEVELOPER_AUTH,
    label="Test Case 6. View target defect details while Open",
)


## Test Case 7: Take Ownership of Open Defect

Purpose: Demonstrate `US-08` and `US-09` from `BT-UC4`.

Actor: Developer

Input:
- Action: `take_ownership`
- Authenticated developer: `dev-001`

Expected: API returns `200` and the target defect status becomes `Assigned` with assignee `dev-001`.

In [ ]:
step7 = curl_json(
    "POST",
    f"/api/defects/{target_report_id}/actions/",
    auth=DEVELOPER_AUTH,
    payload={"action": "take_ownership"},
    label="Test Case 7. Take ownership of target defect report",
)


## Test Case 8: List Assigned Defects

Purpose: Confirm the target defect has moved into the `Assigned` state.

Actor: Developer

Request:
- Endpoint: `GET /api/defects/?status=Assigned`
- Auth: `dev-001`

Expected: Response contains assigned defects for the developer's product, including the target defect.

In [ ]:
step8 = curl_json(
    "GET",
    "/api/defects/",
    auth=DEVELOPER_AUTH,
    params={"status": "Assigned"},
    label="Test Case 8. List Assigned defects for Developer",
)
step8_target = show_target_from_list(step8, target_report_id, "Test Case 8. Confirm target defect in Assigned list")


## Test Case 9: Mark Assigned Defect as Fixed

Purpose: Demonstrate `US-10` and `US-11` from `BT-UC5`.

Actor: Developer

Input:
- Action: `set_fixed`
- Fix note: `Patched for Sprint 1 demo.`

Expected: API returns `200` and the target defect status becomes `Fixed`.

In [ ]:
step9 = curl_json(
    "POST",
    f"/api/defects/{target_report_id}/actions/",
    auth=DEVELOPER_AUTH,
    payload={
        "action": "set_fixed",
        "fix_note": "Patched for Sprint 1 demo.",
    },
    label="Test Case 9. Set target defect report as Fixed",
)


## Test Case 10: List Fixed Defects

Purpose: Demonstrate `US-12` from `BT-UC6`.

Actor: Product Owner

Request:
- Endpoint: `GET /api/defects/?status=Fixed`
- Auth: `owner-001`

Expected: Response contains fixed defects for the product owner's product, including the target defect.

In [ ]:
step10 = curl_json(
    "GET",
    "/api/defects/",
    auth=OWNER_AUTH,
    params={"status": "Fixed"},
    label="Test Case 10. List Fixed defects for Product Owner",
)
step10_target = show_target_from_list(step10, target_report_id, "Test Case 10. Confirm target defect in Fixed list")


## Test Case 11: Resolve Fixed Defect

Purpose: Demonstrate `US-13` and `US-14` from `BT-UC6`.

Actor: Product Owner

Input:
- Action: `set_resolved`
- Retest note: `Verified during Sprint 1 demo.`

Expected: API returns `200` and the target defect status becomes `Resolved`.

In [ ]:
step11 = curl_json(
    "POST",
    f"/api/defects/{target_report_id}/actions/",
    auth=OWNER_AUTH,
    payload={
        "action": "set_resolved",
        "retest_note": "Verified during Sprint 1 demo.",
    },
    label="Test Case 11. Close target defect report as Resolved",
)


## Test Case 12: Final Verification of Resolved Defect (Extra test case)

Purpose: Final confirmation that the target defect is visible in the `Resolved` list.

Actor: Product Owner

Request:
- Endpoint: `GET /api/defects/?status=Resolved`
- Auth: `owner-001`

Expected: Response contains the target defect with status `Resolved`.

In [ ]:
step12 = curl_json(
    "GET",
    "/api/defects/",
    auth=OWNER_AUTH,
    params={"status": "Resolved"},
    label="Test Case 12. List Resolved defects for Product Owner",
)
step12_target = show_target_from_list(step12, target_report_id, "Test Case 12. Confirm target defect in Resolved list")
